In [7]:
import os
from typing import Literal

MODE: Literal["DEBUG", "TRAIN", "EVAL"] = "DEBUG"

In [1]:
from pathlib import Path
import os
import sys

# Base project directory (root of your local project)
PROJECT_ROOT = Path(".")  # current folder where notebook is located
sys.path.append(str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw_dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNK_DIR = PROCESSED_DIR / "chunks"
SPLIT_DIR = PROCESSED_DIR / "splits"
EMB_DIR = PROCESSED_DIR / "embeddings"
INDEX_DIR = PROJECT_ROOT / "indexes"

# Create directories if they do not exist yet
for p in [RAW_DIR, PROCESSED_DIR, CHUNK_DIR, SPLIT_DIR, EMB_DIR, INDEX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Working directories ready:")
print("RAW_DIR      =", RAW_DIR)
print("PROCESSED_DIR=", PROCESSED_DIR)
print("SPLIT_DIR    =", SPLIT_DIR)
print("CHUNK_DIR    =", CHUNK_DIR)
print("EMB_DIR      =", EMB_DIR)
print("INDEX_DIR    =", INDEX_DIR)


Working directories ready:
RAW_DIR      = data\raw_dataset
PROCESSED_DIR= data\processed
SPLIT_DIR    = data\processed\splits
CHUNK_DIR    = data\processed\chunks
EMB_DIR      = data\processed\embeddings
INDEX_DIR    = indexes


In [3]:
from pathlib import Path
from src.data_prep.load_dataset import prepare_full_dataset

raw_train_parquet = RAW_DIR / "train.parquet"

# use file directly if it already exists
if raw_train_parquet.exists():
    print(f"Found existing raw train dataset at {raw_train_parquet}, skipping download.")
    paths = {"train": raw_train_parquet}
else:
    print("No raw train dataset found, preparing full dataset...")
    paths = prepare_full_dataset(
        output_dir=RAW_DIR,
        dataset_name="trivia_qa",
        subset="rc.wikipedia",
        splits=("train",),
        # max_examples_per_split={"train": 50},  # optional for debugging
    )

raw_train_path = paths["train"]
print(f"Using raw_train_path = {raw_train_path}")

No raw train dataset found, preparing full dataset...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Using raw_train_path = data\raw_dataset\train.parquet


In [4]:
from pathlib import Path
from src.data_prep.split_dataset import create_train_val_split

split_train_path = Path(SPLIT_DIR) / "train.parquet"
split_val_path = Path(SPLIT_DIR) / "val_7900.parquet"
split_meta_path = Path(SPLIT_DIR) / "splits_meta.json"

# check if splits already exist
if split_train_path.exists() and split_val_path.exists():
    print("Found existing splits, skipping split creation.")
    split_paths = {
        "train": split_train_path,
        "val": split_val_path,
        "meta": split_meta_path if split_meta_path.exists() else None,
    }
else:
    print("No existing splits found, creating new train/val split...")
    split_paths = create_train_val_split(
        raw_train_path=raw_train_path,
        output_dir=SPLIT_DIR,
        val_size=7900,
        strategy="first_n",
    )

print("Split paths:")
for k, v in split_paths.items():
    print(f"  {k}: {v}")

No existing splits found, creating new train/val split...
Split paths:
  train: data\processed\splits\train.parquet
  val: data\processed\splits\val_7900.parquet
  meta: data\processed\splits\splits_meta.json


In [5]:
if MODE == "DEBUG":
  import pandas as pd
  from pathlib import Path

  val_path = Path(split_paths["val"])
  df_val = pd.read_parquet(val_path)

  print("Validation shape:", df_val.shape)

  expected_val_size = 7900
  if df_val.shape[0] != expected_val_size:
      raise ValueError(
          f"Expected {expected_val_size} validation examples, "
          f"but found {df_val.shape[0]}"
      )
  else:
      print(f"Validation split has the expected size of {expected_val_size} rows.")

  train_path = Path(split_paths["train"])
  df_train = pd.read_parquet(train_path)

  print("Train shape:", df_train.shape)

  # print a random example
  rand_idx = df_train.sample(1).index[0]
  print(f"\n--- Random example (index={rand_idx}) ---")
  display(df_train.loc[rand_idx])

Validation shape: (7900, 7)
Validation split has the expected size of 7900 rows.
Train shape: (53988, 7)

--- Random example (index=27664) ---


orig_index                                                            35564
question_id                                                        sfq_8154
question                  Which 19th century writer wrote a collection o...
answer_aliases_json       ["Oscar Wild", "Flahertie", "C. 3. 3. 3", "Osc...
answer_normalized_json    ["o flahertie", "theocritus villanelle", "flah...
doc_titles_json                        ["The Happy Prince and Other Tales"]
evidence_text             The Happy Prince and Other Tales (sometimes ca...
Name: 27664, dtype: object

In [11]:
from pathlib import Path
from src.data_prep.chunk_documents import chunk_dataset

train_split_path = Path(SPLIT_DIR) / "train.parquet"
val_split_path = Path(SPLIT_DIR) / "val_7900.parquet"

train_chunks_path = Path(CHUNK_DIR) / "train_chunks.parquet"
val_chunks_path = Path(CHUNK_DIR) / "val_chunks.parquet"

# Train-Split chunking
if train_chunks_path.exists():
    print(f"Train chunks already exist at {train_chunks_path}, skipping.")
else:
    print("Creating train chunks...")
    chunk_dataset(
        split_path=train_split_path,
        out_path=train_chunks_path,
        split_name="train",
        text_column="evidence_text",
        title_column="doc_titles_json",
        example_id_column="question_id",
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_size_tokens=512,
        chunk_overlap_tokens=128,
    )

# Val-Split chunking
if val_chunks_path.exists():
    print(f"Validation chunks already exist at {val_chunks_path}, skipping.")
else:
    print("Creating validation chunks...")
    chunk_dataset(
        split_path=val_split_path,
        out_path=val_chunks_path,
        split_name="val",
        text_column="evidence_text",
        title_column="doc_title",
        example_id_column="question_id",
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_size_tokens=512,
        chunk_overlap_tokens=128,
    )

Creating train chunks...
Neue Version funktioniert!


Chunking train:   0%|          | 0/53988 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10705 > 512). Running this sequence through the model will result in indexing errors


Creating validation chunks...
Neue Version funktioniert!


Chunking val:   0%|          | 0/7900 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (24919 > 512). Running this sequence through the model will result in indexing errors


In [12]:
if MODE == "DEBUG":
  from pathlib import Path
  import pandas as pd
  from transformers import AutoTokenizer

  # === CONFIG ===
  CHUNKS_PATH = train_chunks_path  # oder val_chunks_path
  MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
  CHUNK_SIZE = 512
  OVERLAP = 128

  # === LOAD CHUNKS ===
  df_chunks = pd.read_parquet(CHUNKS_PATH)
  print(f"Loaded {len(df_chunks)} chunks.")
  print(f"\nStructure\n:{df_chunks.info()}")

  # === Inspect one example_id with multiple chunks ===
  example_id = df_chunks["example_id"].iloc[0]  # first example in dataset
  example_chunks = df_chunks[df_chunks["example_id"] == example_id].sort_values("start_token")

  print("\nExample ID:", example_id)
  print("Number of chunks for this example:", len(example_chunks))

  # === Show first chunk text ===
  print("\n=== First chunk snippet ===")
  print(example_chunks.iloc[0]["chunk_text"][:500], "...\n")

  # === Verify token lengths and overlaps ===
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

  print("=== Checking token lengths and overlaps ===")
  for i, row in example_chunks.iterrows():
      tokens = tokenizer.encode(
          row["chunk_text"],
          add_special_tokens=False
      )
      length = len(tokens)
      print(f"Chunk {row['chunk_id']}: {length} tokens")

  print("\nExpected:")
  print(f"- Typical chunk length: ~{CHUNK_SIZE} tokens")
  print(f"- Overlap:             ~{OVERLAP} tokens")

  # === Check boundaries explicitly ===
  print("\n=== Checking token boundary consistency ===")
  for idx in range(len(example_chunks) - 1):
      curr = example_chunks.iloc[idx]
      nxt = example_chunks.iloc[idx + 1]
      overlap_real = curr["end_token"] - nxt["start_token"]
      print(
          f"Between chunk {idx} and {idx+1}: "
          f"expected overlap {OVERLAP}, real overlap {overlap_real}"
      )

  # === Optional: Show a random chunk pair to visually inspect ===
  import random
  sample_rows = example_chunks.sample(2)
  print("\n=== Random chunk sample texts ===")
  for _, row in sample_rows.iterrows():
      print(f"\nChunk ID: {row['chunk_id']}")
      print(row["chunk_text"][:400], "...")

Loaded 1597840 chunks.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1597840 entries, 0 to 1597839
Data columns (total 7 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   chunk_id     1597840 non-null  object
 1   split        1597840 non-null  object
 2   example_id   1597840 non-null  object
 3   doc_title    1597840 non-null  object
 4   chunk_text   1597840 non-null  object
 5   start_token  1597840 non-null  int64 
 6   end_token    1597840 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 85.3+ MB

Structure
:None

Example ID: qb_6731
Number of chunks for this example: 28

=== First chunk snippet ===
the mediterranean sea ( pronounced ) is a sea connected to the atlantic ocean surrounded by the mediterranean basin and almost completely enclosed by land : on the north by southern europe and anatolia, on the south by north africa, and on the east by the levant. the sea is sometimes considered a part of the atlantic oce

Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


=== Checking token lengths and overlaps ===
Chunk train_qb_6731_0: 512 tokens
Chunk train_qb_6731_1: 512 tokens
Chunk train_qb_6731_2: 512 tokens
Chunk train_qb_6731_3: 512 tokens
Chunk train_qb_6731_4: 512 tokens
Chunk train_qb_6731_5: 512 tokens
Chunk train_qb_6731_6: 512 tokens
Chunk train_qb_6731_7: 512 tokens
Chunk train_qb_6731_8: 512 tokens
Chunk train_qb_6731_9: 512 tokens
Chunk train_qb_6731_10: 512 tokens
Chunk train_qb_6731_11: 512 tokens
Chunk train_qb_6731_12: 512 tokens
Chunk train_qb_6731_13: 512 tokens
Chunk train_qb_6731_14: 512 tokens
Chunk train_qb_6731_15: 512 tokens
Chunk train_qb_6731_16: 512 tokens
Chunk train_qb_6731_17: 513 tokens
Chunk train_qb_6731_18: 512 tokens
Chunk train_qb_6731_19: 512 tokens
Chunk train_qb_6731_20: 512 tokens
Chunk train_qb_6731_21: 512 tokens
Chunk train_qb_6731_22: 514 tokens
Chunk train_qb_6731_23: 512 tokens
Chunk train_qb_6731_24: 512 tokens
Chunk train_qb_6731_25: 512 tokens
Chunk train_qb_6731_26: 512 tokens
Chunk train_qb_6731_2

In [28]:
example_id = "qw_9222"  # Beispiel
df_chunks[df_chunks['example_id'] == example_id].head()

,chunk_id,split,example_id,doc_title,chunk_text,start_token,end_token
519852,train_qw_9222_0,train,qw_9222,First Lady,first lady is an unofficial title used for the...,0,512
519853,train_qw_9222_1,train,qw_9222,First Lady,men. he thus took the more humble title prince...,384,896
519854,train_qw_9222_2,train,qw_9222,First Lady,"hostess in the white house, if the president's...",768,1280
519855,train_qw_9222_3,train,qw_9222,First Lady,"said of carmen romero rubio de diaz : "" she is...",1152,1664
519856,train_qw_9222_4,train,qw_9222,First Lady,to the wife of the taoiseach ( prime minister ...,1536,2048


In [ ]:
from src.data_prep.embed_documents import compute_and_save_embeddings

EMB_DIR = PROJECT_ROOT / "data" / "processed" / "embeddings"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks" / "train_chunks.parquet"

compute_and_save_embeddings(
    base_dir=EMB_DIR,
    chunks_parquet_path=CHUNKS_PATH,
)

Using device: cuda
Embeddings not found, computing embeddings...


Loading chunks from parquet: 100%|██████████| 1597840/1597840 [01:34<00:00, 16882.36it/s]


Loaded 1597840 chunks
Removed 636228 duplicates → 961612 unique passages.


Embedding: 100%|██████████| 1879/1879 [1:05:03<00:00,  2.08s/it]


Saved embeddings to data\processed\embeddings\corpus_embeddings_unique.pkl


(['Mediterranean Sea: the mediterranean sea ( pronounced ) is a sea connected to the atlantic ocean surrounded by the mediterranean basin and almost completely enclosed by land : on the north by southern europe and anatolia, on the south by north africa, and on the east by the levant. the sea is sometimes considered a part of the atlantic ocean, although it is usually identified as a separate body of water. the name mediterranean is derived from the latin mediterraneus, meaning " inland " or " in the middle of land " ( from medius, " middle " and terra, " land " ). it covers an approximate area of 2. 5 million km2 ( 965, 000 sq mi ), but its connection to the atlantic ( the strait of gibraltar ) is only 14 km wide. the strait of gibraltar is a narrow strait that connects the atlantic ocean to the mediterranean sea and separates gibraltar and spain in europe from morocco in africa. in oceanography, it is sometimes called the eurafrican mediterranean sea or the european mediterranean sea